In [9]:
import tkinter as tk
from tkinter import simpledialog
from PIL import Image, ImageTk, ImageOps
import os
os.chdir(r"C:\Users\jdp8\Documents\wa")

# Create window
window = tk.Tk()
window.title("Jazmin's Pet Daycare")
window.geometry("800x600")
window.configure(bg="pink")

# Pet needs and coins
hunger = 100
energy = 100
happiness = 100
coins = 0

# Load pet images
cat_image = Image.open("images/cat.png")
cat_image = ImageOps.contain(cat_image, (250, 250))

# Make cat black and white but keep transparent background
cat_image = cat_image.convert("RGBA")
gray_cat = ImageOps.grayscale(cat_image).convert("RGBA")
gray_cat.putalpha(cat_image.getchannel("A"))
cat_photo = ImageTk.PhotoImage(gray_cat, master=window)

dog_image = Image.open("images/dog.png")
dog_image = ImageOps.contain(dog_image, (250, 250))
dog_photo = ImageTk.PhotoImage(dog_image, master=window)

dog_hungry = Image.open("images/dog hungry.png")
dog_hungry = ImageOps.contain(dog_hungry, (350, 350))
dog_hungry_photo = ImageTk.PhotoImage(dog_hungry, master=window)

dog_sleepy = Image.open("images/dog sleepy.png")
dog_sleepy = ImageOps.contain(dog_sleepy, (350, 350))
dog_sleepy_photo = ImageTk.PhotoImage(dog_sleepy, master=window)

dog_sad = Image.open("images/dog sad.png")
dog_sad = ImageOps.contain(dog_sad, (350, 350))
dog_sad_photo = ImageTk.PhotoImage(dog_sad, master=window)

# Load activity icons
sleep_image = Image.open("images/sleep.png")
sleep_image = ImageOps.contain(sleep_image, (140, 140))
sleep_photo = ImageTk.PhotoImage(sleep_image, master=window)

eat_image = Image.open("images/eat.png")
eat_image = ImageOps.contain(eat_image, (140, 140))
eat_photo = ImageTk.PhotoImage(eat_image, master=window)

play_image = Image.open("images/play.png")
play_image = ImageOps.contain(play_image, (140, 140))
play_photo = ImageTk.PhotoImage(play_image, master=window)


# Coin counter - visible on both screens
coin_label = tk.Label(
    window,
    text="🪙 " + str(coins),
    font=("Trebuchet MS", 22, "bold"),
    bg="pink",
    fg="white"
)
coin_label.place(relx=0.04, rely=0.94, anchor="w")


# Check dog's mood
def check_mood():
    if hunger <= 30:
        pet_label.config(image=dog_hungry_photo)
        mood_label.config(
            text=pet_name + " is hungry. Feed " + pet_name + "!"
        )

    elif energy <= 30:
        pet_label.config(image=dog_sleepy_photo)
        mood_label.config(
            text=pet_name + " is sleepy. Let " + pet_name + " rest!"
        )

    elif happiness <= 30:
        pet_label.config(image=dog_sad_photo)
        mood_label.config(
            text=pet_name + " is sad. Play with " + pet_name + "!"
        )

    else:
        pet_label.config(image=dog_photo)
        mood_label.config(text=pet_name + " is happy!")


# Decrease needs every 5 seconds
def update_mood():
    global hunger, energy, happiness, mood_timer

    hunger = max(0, hunger - 10)
    energy = max(0, energy - 5)
    happiness = max(0, happiness - 7)

    check_mood()

    mood_timer = window.after(5000, update_mood)


# Pet activities
def feed_pet():
    global hunger
    hunger = 100
    check_mood()


def sleep_pet():
    global energy
    energy = 100
    check_mood()


def play_pet():
    global happiness
    happiness = 100
    check_mood()


# Earn one coin
def earn_coin():
    global coins

    coins += 1
    coin_label.config(text="🪙 " + str(coins))


# Start dragging
def start_drag(event):
    global dragged_image

    button = event.widget

    dragged_image = tk.Label(
        window,
        image=button["image"],
        bg="pink",
        borderwidth=0
    )

    x = event.x_root - window.winfo_rootx()
    y = event.y_root - window.winfo_rooty()

    dragged_image.place(x=x, y=y, anchor="center")
    dragged_image.lift()


# Move icon
def drag_icon(event):
    x = event.x_root - window.winfo_rootx()
    y = event.y_root - window.winfo_rooty()

    dragged_image.place(x=x, y=y, anchor="center")


# Drop icon onto dog
def stop_drag(event):
    x = event.x_root
    y = event.y_root

    dog_x = pet_label.winfo_rootx()
    dog_y = pet_label.winfo_rooty()

    dog_width = pet_label.winfo_width()
    dog_height = pet_label.winfo_height()

    if dog_x <= x <= dog_x + dog_width and dog_y <= y <= dog_y + dog_height:

        if event.widget == feed_button:
            feed_pet()
            earn_coin()

        elif event.widget == sleep_button:
            sleep_pet()
            earn_coin()

        elif event.widget == play_button:
            play_pet()
            earn_coin()

    dragged_image.destroy()

def return_to_menu():
    global hunger, energy, happiness, mood_timer

    # Stop the dog's timer
    window.after_cancel(mood_timer)

    # Reset pet needs
    hunger = 100
    energy = 100
    happiness = 100

    # Remove game screen
    salon_message.destroy()
    name_label.destroy()
    game_frame.destroy()

    sleep_button.destroy()
    feed_button.destroy()
    play_button.destroy()
    menu_button.destroy()

    # Bring back the main menu
    title.pack(pady=30)
    welcome.pack(pady=40)
    button_frame.pack(pady=50)

# Select dog
def select_pet():
    global salon_message, name_label, game_frame, menu_button, mood_timer
    global sleep_button, feed_button, play_button
    global pet_label, mood_label, pet_name

    pet_name = simpledialog.askstring(
        "Name Your Pet",
        "What would you like to name your dog?"
    )

    if pet_name:
        menu_button = tk.Button(
            window,
            text="← Main Menu",
            command=return_to_menu,
            font=("Trebuchet MS", 13, "bold"),
            bg="#FF69B4",
            fg="white",
            activebackground="#FF1493",
            activeforeground="white",
            borderwidth=0,
            padx=15,
            pady=8,
            cursor="hand2"
        )

        menu_button.place(relx=0.03, rely=0.50, anchor="w")

        title.pack_forget()
        welcome.pack_forget()
        button_frame.pack_forget()

        # Welcome message
        salon_message = tk.Label(
            window,
            text="Welcome to your pet daycare!",
            font=("Trebuchet MS", 25, "bold"),
            bg="pink",
            fg="white"
        )
        salon_message.pack(pady=(20, 5))

        # Pet name
        name_label = tk.Label(
            window,
            text=pet_name,
            font=("Trebuchet MS", 20, "bold"),
            bg="pink",
            fg="white"
        )
        name_label.pack(pady=5)

        # Main game area
        game_frame = tk.Frame(window, bg="pink")
        game_frame.pack(pady=10)

        pet_frame = tk.Frame(game_frame, bg="pink")
        pet_frame.pack(side="left", padx=30)

        # Dog picture
        pet_label = tk.Label(
            pet_frame,
            image=dog_photo,
            bg="pink"
        )
        pet_label.pack()

        # Emotion message
        mood_label = tk.Label(
            pet_frame,
            text=pet_name + " is happy!",
            font=("Trebuchet MS", 16, "bold"),
            bg="pink",
            fg="white"
        )
        mood_label.pack(pady=10)

        # Activity buttons
        sleep_button = tk.Button(
            window,
            image=sleep_photo,
            borderwidth=0,
            bg="pink"
        )
        sleep_button.place(relx=0.95, rely=0.30, anchor="e")

        feed_button = tk.Button(
            window,
            image=eat_photo,
            borderwidth=0,
            bg="pink"
        )
        feed_button.place(relx=0.95, rely=0.50, anchor="e")

        play_button = tk.Button(
            window,
            image=play_photo,
            borderwidth=0,
            bg="pink"
        )
        play_button.place(relx=0.95, rely=0.70, anchor="e")

        # Dragging controls
        sleep_button.bind("<Button-1>", start_drag)
        sleep_button.bind("<B1-Motion>", drag_icon)
        sleep_button.bind("<ButtonRelease-1>", stop_drag)

        feed_button.bind("<Button-1>", start_drag)
        feed_button.bind("<B1-Motion>", drag_icon)
        feed_button.bind("<ButtonRelease-1>", stop_drag)

        play_button.bind("<Button-1>", start_drag)
        play_button.bind("<B1-Motion>", drag_icon)
        play_button.bind("<ButtonRelease-1>", stop_drag)

        # Start mood timer
        mood_timer = window.after(5000, update_mood)


# Selection screen
title = tk.Label(
    window,
    text="Welcome to Jazmin's Pet Daycare!",
    font=("Trebuchet MS", 35, "bold"),
    bg="pink",
    fg="white"
)
title.pack(pady=30)

welcome = tk.Label(
    window,
    text="Choose a pet!",
    font=("Georgia", 25),
    bg="pink",
    fg="white"
)
welcome.pack(pady=40)

# Pet selection frames
button_frame = tk.Frame(window, bg="pink")
button_frame.pack(pady=50)

cat_frame = tk.Frame(button_frame, bg="pink")
cat_frame.pack(side="left", padx=15)

dog_frame = tk.Frame(button_frame, bg="pink")
dog_frame.pack(side="left", padx=15)

# Locked cat
cat_text = tk.Label(
    cat_frame,
    text="🔒 Unlock for 1,000 coins",
    font=("Trebuchet MS", 15, "bold"),
    bg="pink",
    fg="white"
)
cat_text.pack(pady=5)

cat_label = tk.Label(
    cat_frame,
    image=cat_photo,
    bg="pink"
)
cat_label.pack()

# Free dog
dog_text = tk.Label(
    dog_frame,
    text="DOG - FREE",
    font=("Trebuchet MS", 16, "bold"),
    bg="pink",
    fg="white"
)
dog_text.pack(pady=5)

dog_button = tk.Button(
    dog_frame,
    image=dog_photo,
    command=select_pet,
    borderwidth=0,
    bg="pink"
)
dog_button.pack()

window.mainloop()
